# ProjectMind AI — Google Colab Runtime

Run cells in order. This notebook starts Ollama, installs the backend, launches FastAPI and exposes it through ngrok. Keep the runtime running while using the HTML frontend.

## 1. Mount Google Drive and install dependencies

Drive stores ChromaDB and repository data across runtime restarts. Runtime availability and Drive persistence are not guaranteed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DATA_DIR = '/content/drive/MyDrive/ProjectMindAI'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)
os.environ['PROJECTMIND_DATA'] = DATA_DIR


In [ ]:
!pip -q install fastapi uvicorn python-multipart pyngrok langchain langchain-core langchain-chroma langchain-ollama chromadb pydantic
!apt-get -qq update && apt-get -qq install -y curl


## 2. Install and start Ollama

The model download may take several minutes. If the runtime has limited RAM, keep the small Qwen model or choose another Ollama model that fits.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import subprocess, time, requests, os
ollama_log = open('/content/ollama.log', 'w')
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=ollama_log, stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        requests.get('http://127.0.0.1:11434/api/tags', timeout=2)
        break
    except Exception:
        time.sleep(2)
print("Ollama server ready")


In [ ]:
!ollama pull nomic-embed-text
!ollama pull qwen2.5-coder:1.5b


## 3. Upload the backend app.py

From the downloaded ProjectMind AI ZIP, upload `backend/app.py` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select backend/app.py
assert 'app.py' in uploaded, "Please upload the backend app.py file"
with open('/content/app.py', 'wb') as f:
    f.write(uploaded['app.py'])
print("Backend source saved.")


## 4. Configure API key and launch FastAPI

In [ ]:
import secrets, os
API_KEY = secrets.token_urlsafe(32)
os.environ['PROJECTMIND_API_KEY'] = API_KEY
os.environ['PROJECTMIND_DATA'] = DATA_DIR
os.environ['OLLAMA_BASE_URL'] = 'http://127.0.0.1:11434'
os.environ['EMBED_MODEL'] = 'nomic-embed-text'
os.environ['CHAT_MODEL'] = 'qwen2.5-coder:1.5b'
print("ProjectMind API key (copy it into the frontend; do not share it):")
print(API_KEY)


In [ ]:
import threading, uvicorn, time
from importlib.util import spec_from_file_location, module_from_spec
spec = spec_from_file_location("projectmind_app", "/content/app.py")
projectmind_app = module_from_spec(spec)
spec.loader.exec_module(projectmind_app)

config = uvicorn.Config(projectmind_app.app, host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
server_thread = threading.Thread(target=server.run, daemon=True)
server_thread.start()
time.sleep(3)
print("FastAPI started on port 8000")


## 5. Expose API with ngrok

Create an ngrok account and copy your authtoken from its dashboard. Do not publish the token.

In [ ]:
from pyngrok import ngrok
from getpass import getpass

NGROK_TOKEN = getpass("Paste ngrok authtoken: ")
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000, "http")
API_URL = tunnel.public_url
print("ProjectMind API URL:", API_URL)
print("Health check:", API_URL + "/health")
print("Paste API URL and the API key from Step 4 into the frontend Connection settings.")


## 6. Verify API

Open the health URL in a browser. The `/health` endpoint is public; project endpoints require the API key.

In [ ]:
import requests
r = requests.get(API_URL + '/health', timeout=20)
print(r.status_code, r.json())


## 7. Use the frontend

Open `frontend/index.html` from the extracted ZIP in your browser. In Connection settings, paste the API URL and API key. Add a ZIP repository or a public GitHub repository, click Analyze, then ask questions. Keep this Colab runtime active.

In [ ]:
print("Runtime is ready. Keep this notebook session running while using ProjectMind AI.")
